In [0]:
%run ../00-common/01.environment-config

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
fact_path = f"{s3_root_path}/Gold/Aaisha/fact_session_results"

constructors_path = f"{s3_root_path}/Gold/Aaisha/dim_constructors"

target_path = f"{s3_root_path}/Gold/Aaisha/constructor_standings"

In [0]:
fact_df = (
    spark.read
    .format("delta")
    .load(fact_path)
)

constructors_df = (
    spark.read
    .format("delta")
    .load(constructors_path)
)

display(fact_df.limit(10))

In [0]:
constructor_points_df = (
    fact_df
    .groupBy(
        "season",
        "constructorId"
    )
    .agg(
        F.sum("points").alias("total_points")
    )
)

display(constructor_points_df)

In [0]:
window_spec = (
    Window
    .partitionBy("season")
    .orderBy(F.desc("total_points"))
)

constructor_standings_df = (
    constructor_points_df
    .withColumn(
        "constructor_rank",
        F.dense_rank().over(window_spec)
    )
)

display(constructor_standings_df)

In [0]:
constructor_standings_with_name_df = (
    constructor_standings_df
    .join(
        constructors_df.select(
            "constructor_id",
            "constructor_name"
        ),
        constructor_standings_df.constructorId == constructors_df.constructor_id,
        "left"
    )
)

display(constructor_standings_with_name_df)

In [0]:
constructor_standings_final_df = (
    constructor_standings_with_name_df
    .select(
        "season",
        "constructor_rank",
        "constructor_id",
        "constructor_name",
        "total_points"
    )
    .orderBy(
        "season",
        "constructor_rank"
    )
)

display(constructor_standings_final_df)

In [0]:
(
    constructor_standings_final_df
    .write
    .format("delta")
    .mode("overwrite")
    .save(target_path)
)

print("Constructor standings successfully created")

In [0]:
constructor_check_df = (
    spark.read
    .format("delta")
    .load(target_path)
)

display(constructor_check_df)

In [0]:
constructor_race_points_df = (
    fact_df
    .filter(F.col("session_type") == "RACE")
    .groupBy(
        "season",
        "round",
        "constructorId"
    )
    .agg(
        F.sum("points").alias("race_points")
    )
)

display(constructor_race_points_df)

In [0]:
points_window = (
    Window
    .partitionBy("season", "constructorId")
    .orderBy("round")
)

constructor_cumulative_df = (
    constructor_race_points_df
    .withColumn(
        "total_points",
        F.sum("race_points").over(points_window)
    )
)

display(constructor_cumulative_df)

In [0]:
rank_window = (
    Window
    .partitionBy("season", "round")
    .orderBy(F.desc("total_points"))
)

constructor_race_standings_df = (
    constructor_cumulative_df
    .withColumn(
        "constructor_rank",
        F.dense_rank().over(rank_window)
    )
)

display(constructor_race_standings_df)

In [0]:
constructor_race_final_df = (
    constructor_race_standings_df
    .join(
        constructors_df.select(
            "constructor_id",
            "constructor_name"
        ),
        constructor_race_standings_df.constructorId == constructors_df.constructor_id,
        "left"
    )
)

display(constructor_race_final_df)

In [0]:
constructor_race_final_df = (
    constructor_race_final_df
    .select(
        "season",
        "round",
        "constructor_rank",
        "constructor_id",
        "constructor_name",
        "race_points",
        "total_points"
    )
    .orderBy(
        "season",
        "round",
        "constructor_rank"
    )
)

display(constructor_race_final_df)

In [0]:
race_target_path = f"{s3_root_path}/Gold/Aaisha/constructor_race_standings"

(
    constructor_race_final_df
    .write
    .format("delta")
    .mode("overwrite")
    .save(race_target_path)
)

print("Constructor race standings successfully created")

In [0]:
constructor_race_check_df = (
    spark.read
    .format("delta")
    .load(race_target_path)
)

display(constructor_race_check_df)